In [1]:
from langchain_ollama import ChatOllama
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
import pandas as pd
from langchain_chroma import Chroma
import torch, torchvision
print(torch.__version__, torch.version.cuda)
print(torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

2.14.1+cpu None
0.29.1+cu126
CUDA available: False


In [ ]:
import sys
import subprocess
import torch

print(sys.executable)


s:\Projects\rag-dissertation\.venv\Scripts\python.exe


: 

In [2]:
#importing dataset

df = pd.read_csv("../datasets/pubmed abstracts/pubmed_abstracts copy.csv")
df.head()

,Unnamed: 0,deep_learning
0,0,(['Magnetic resonance spectroscopic imaging (M...
1,1,(['Existing deep convolutional neural networks...
2,2,(['Deep learning techniques have been increasi...
3,3,(['The original article unfortunately containe...
4,4,(['The most common applications of artificial ...


In [3]:
#loading dataset in langchain

docs = []
for col in df.columns:
    if col == "Unnamed: 0":
        continue
    text = "\n".join(df[col].dropna().astype(str))
    docs.append(Document(page_content=text, metadata={"column": col}))

print(len(docs))   # = number of columns
print(docs[0].metadata)
print(type(docs[0].page_content))

1
{'column': 'deep_learning'}
<class 'str'>


In [4]:
#Spliting text

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
texts = text_splitter.split_documents(docs)
print(len(texts)) 
print(texts[5].page_content)

22842
by the classification branch. Feeding a chest radiograph to the trained Thorax-Net, a diagnosis is obtained by averaging and binarizing the outputs of two branches. The proposed Thorax-Net model has been evaluated against three state-of-the-art deep learning models using the patientwise official split of the ChestX-ray14 dataset and against other five deep learning models using the imagewise random data split. Our results show that Thorax-Net achieves an average per-class area under the receiver operating characteristic curve (AUC) of 0.7876 and 0.896 in both experiments, respectively, which are higher than the AUC values obtained by other deep models when they were all trained with no external data.'], 'Thorax-Net: An Attention Regularized Deep Neural Network for Classification of Thoracic Diseases on Chest Radiography.')


In [5]:
#Creating Embaddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"batch_size": 8},
    )

query_embedding = embeddings.embed_query("What is a sentence embedding?")
doc_embeddings = embeddings.embed_documents(
    [
        "Sentence embeddings map text to dense vectors.",
        "LangChain provides a standard Embeddings interface.",
    ]
)

s:\Projects\rag-dissertation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ValueError: Unable to compare versions for numpy>=1.17: need=1.17 found=None. This is unusual. Consider reinstalling numpy.

In [16]:
#Vector database

vector_store = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db",  # Where to save data locally, remove if not necessary
)

vector_store.add_documents(texts)

OutOfMemoryError: CUDA out of memory. Tried to allocate 3.38 GiB. GPU 0 has a total capacity of 4.00 GiB of which 0 bytes is free. Process 1852 has 17179869184.00 GiB memory in use. Process 8296 has 17179869184.00 GiB memory in use. Process 9208 has 17179869184.00 GiB memory in use. Process 7964 has 17179869184.00 GiB memory in use. Process 13308 has 17179869184.00 GiB memory in use. Process 14120 has 17179869184.00 GiB memory in use. Process 12900 has 17179869184.00 GiB memory in use. Process 13752 has 17179869184.00 GiB memory in use. Process 3480 has 17179869184.00 GiB memory in use. Process 11204 has 17179869184.00 GiB memory in use. Process 15204 has 17179869184.00 GiB memory in use. Process 15268 has 17179869184.00 GiB memory in use. Process 14116 has 17179869184.00 GiB memory in use. Process 14268 has 17179869184.00 GiB memory in use. Process 7140 has 17179869184.00 GiB memory in use. Process 19428 has 17179869184.00 GiB memory in use. Process 18688 has 17179869184.00 GiB memory

In [13]:
#Retrieval

retriever = vector_store.as_retriever()

ret = retriever.invoke("what is adhd?")

In [14]:
print(ret)

[]


In [ ]:
#Setting up LLM

llm = ChatOllama(
    model="gemma4:e4b", 
    )

In [ ]:
#Query LLM

response = llm.invoke("Hey how u doing today")

In [6]:
print(response.text)

I'm doing great, thank you for asking! 😊 Just here and ready to chat.

How about you? What's going on with you today?
